# ₿ Bitcoin y Criptomonedas — Análisis para Mercado LATAM

**Objetivo:** Analizar el comportamiento de Bitcoin y criptomonedas principales desde la perspectiva de un inversor latinoamericano, con énfasis en su rol como activo refugio frente a la depreciación de monedas locales.

**Contexto Bolivia:** Con el dólar paralelo en alza desde 2022 y restricciones cambiarias, Bitcoin ha ganado relevancia como alternativa de ahorro para bolivianos. Este análisis cuantifica esa relación.

**Técnicas aplicadas:**
- Análisis de volatilidad histórica (GARCH-like con rolling std)
- Correlación dinámica BTC vs activos LATAM
- Análisis de drawdown y recuperación
- Ciclos de mercado: bull vs bear market
- Comparación BTC vs inflación boliviana como cobertura

In [ ]:
# ─── DEPENDENCIAS ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf
from scipy import stats
from scipy.stats import norm, jarque_bera
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
SEED = 42

# Paleta cripto
COLORES = {
    'BTC-USD':  '#F7931A',   # Naranja Bitcoin
    'ETH-USD':  '#627EEA',   # Azul Ethereum
    'BNB-USD':  '#F3BA2F',   # Amarillo Binance
    'ILF':      '#2E86C1',   # Azul LATAM ETF
    'GLD':      '#F1C40F',   # Oro
}
print('✅ Librerías cargadas')

## 1. Descarga de Datos

In [ ]:
# ─── DESCARGA ────────────────────────────────────────────────────────────────
TICKERS = {
    'BTC-USD': 'Bitcoin',
    'ETH-USD': 'Ethereum',
    'BNB-USD': 'Binance Coin',
    'ILF':     'iShares LATAM 40',
    'GLD':     'SPDR Gold ETF',
}

INICIO = '2020-01-01'
FIN    = '2024-12-31'

print(f'Descargando {INICIO} → {FIN}...')
raw    = yf.download(list(TICKERS.keys()), start=INICIO, end=FIN, auto_adjust=True)
precios = raw['Close'].dropna()
retornos = np.log(precios / precios.shift(1)).dropna()

print(f'\n✅ {len(precios)} días de datos')
print(f'   Período: {precios.index[0].date()} → {precios.index[-1].date()}')
precios.tail(3)

## 2. Evolución de Precios y Ciclos de Mercado

In [ ]:
# ─── PRECIOS NORMALIZADOS ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 11))

# Panel 1: Todos los activos normalizados a 100
norm_precios = precios / precios.iloc[0] * 100
for ticker in TICKERS:
    color = COLORES.get(ticker, '#888')
    lw    = 2.5 if ticker == 'BTC-USD' else 1.5
    norm_precios[ticker].plot(ax=axes[0], color=color, lw=lw,
                               label=f"{TICKERS[ticker]} ({ticker})")

axes[0].set_title('Evolución Normalizada (Base 100 = Ene 2020)\nCriptos vs Activos Tradicionales',
                  fontweight='bold')
axes[0].set_ylabel('Valor (base 100)')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].set_yscale('log')  # Escala log para ver mejor la amplitud cripto
axes[0].set_ylabel('Valor (escala logarítmica)')

# Panel 2: Solo BTC con zonas bull/bear coloreadas
btc = precios['BTC-USD'].dropna()
btc_norm = btc / btc.iloc[0] * 100
axes[1].plot(btc.index, btc_norm.values, color='#F7931A', lw=2, label='BTC')
axes[1].fill_between(btc.index, btc_norm.values, 100, 
                      where=btc_norm.values >= 100,
                      alpha=0.15, color='#27AE60', label='Por encima de precio inicial')
axes[1].fill_between(btc.index, btc_norm.values, 100,
                      where=btc_norm.values < 100,
                      alpha=0.15, color='#E74C3C', label='Por debajo de precio inicial')

# Anotar máximos históricos
idx_max = btc_norm.idxmax()
axes[1].annotate(f'ATH\n{btc[idx_max]:,.0f}',
                  xy=(idx_max, btc_norm[idx_max]),
                  xytext=(30, -30), textcoords='offset points',
                  fontsize=9, color='#27AE60', fontweight='bold',
                  arrowprops=dict(arrowstyle='->', color='#27AE60'))

axes[1].set_title('Bitcoin — Ciclos Bull/Bear (2020-2024)', fontweight='bold')
axes[1].set_ylabel('Valor (base 100)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('img/01_precios_ciclos.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Análisis de Volatilidad

In [ ]:
# ─── VOLATILIDAD HISTÓRICA ROLLING ───────────────────────────────────────────
# Volatilidad anualizada con ventana de 30 días
WINDOW = 30
vol_rolling = retornos.rolling(WINDOW).std() * np.sqrt(252) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Comparación de volatilidades
for ticker in TICKERS:
    color = COLORES.get(ticker, '#888')
    lw = 2.5 if ticker == 'BTC-USD' else 1.5
    vol_rolling[ticker].plot(ax=axes[0], color=color, lw=lw, alpha=0.85,
                              label=f"{ticker}")

axes[0].set_title(f'Volatilidad Histórica Rolling {WINDOW}d (Anualizada %)', fontweight='bold')
axes[0].set_ylabel('Volatilidad (%)')
axes[0].legend(fontsize=9)

# Comparación estadística
stats_vol = pd.DataFrame({
    'Activo': [TICKERS[t] for t in TICKERS],
    'Vol. Media (%)': (retornos.std() * np.sqrt(252) * 100).round(1).values,
    'Vol. Máx (%)':   (vol_rolling.max()).round(1).values,
    'Vol. Mín (%)':   (vol_rolling.min()).round(1).values,
})

colors_bar = [COLORES.get(t, '#888') for t in TICKERS]
bars = axes[1].bar(stats_vol['Activo'], stats_vol['Vol. Media (%)'],
                    color=colors_bar, alpha=0.85, edgecolor='white')
axes[1].set_title('Volatilidad Media Anualizada por Activo', fontweight='bold')
axes[1].set_ylabel('Volatilidad media (%)')
for bar, val in zip(bars, stats_vol['Vol. Media (%)']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                  f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('img/02_volatilidad.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Comparación de volatilidad:')
print(stats_vol.to_string(index=False))

## 4. Drawdown Analysis

In [ ]:
# ─── DRAWDOWN ────────────────────────────────────────────────────────────────
# Drawdown = caída desde el máximo histórico hasta el mínimo local
# Mide el peor escenario para un inversor que compró en el pico

def calcular_drawdown(precios_serie):
    maximo_acumulado = precios_serie.cummax()
    drawdown = (precios_serie - maximo_acumulado) / maximo_acumulado * 100
    return drawdown

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(14, 16), sharex=True)

for i, ticker in enumerate(TICKERS):
    dd = calcular_drawdown(precios[ticker].dropna())
    color = COLORES.get(ticker, '#888')

    axes[i].fill_between(dd.index, dd.values, 0, alpha=0.4, color=color)
    axes[i].plot(dd.index, dd.values, color=color, lw=1)
    axes[i].axhline(0, color='black', lw=0.8)

    max_dd = dd.min()
    idx_max_dd = dd.idxmin()
    axes[i].annotate(f'{max_dd:.1f}%',
                      xy=(idx_max_dd, max_dd),
                      xytext=(10, 10), textcoords='offset points',
                      fontsize=9, color=color, fontweight='bold')

    axes[i].set_ylabel(f'{ticker}\nDrawdown (%)', fontsize=9)
    axes[i].set_ylim(top=5)

axes[0].set_title('Análisis de Drawdown por Activo\n(Caída desde máximo histórico local)',
                   fontweight='bold')
plt.tight_layout()
plt.savefig('img/03_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📉 Máximos Drawdowns:')
for ticker in TICKERS:
    dd = calcular_drawdown(precios[ticker].dropna())
    print(f'   {ticker:<10}: {dd.min():.1f}% (en {dd.idxmin().date()})')

## 5. Correlación Dinámica con Activos LATAM

In [ ]:
# ─── CORRELACIÓN ROLLING ─────────────────────────────────────────────────────
# ¿BTC se mueve como los mercados LATAM o de forma independiente?
# Correlación rolling de 60 días

WINDOW_CORR = 60

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Correlación BTC vs otros
pares = [('BTC-USD', 'ILF'), ('BTC-USD', 'GLD'), ('ETH-USD', 'BTC-USD')]
colores_pares = ['#E74C3C', '#F1C40F', '#627EEA']

for (t1, t2), color in zip(pares, colores_pares):
    corr_rolling = retornos[t1].rolling(WINDOW_CORR).corr(retornos[t2])
    axes[0].plot(corr_rolling.index, corr_rolling.values, color=color, lw=2,
                  label=f'{t1} vs {t2}')

axes[0].axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
axes[0].axhline(0.5, color='gray', lw=1, linestyle=':', alpha=0.5, label='Umbral correlación alta')
axes[0].axhline(-0.5, color='gray', lw=1, linestyle=':', alpha=0.5)
axes[0].set_title(f'Correlación Rolling {WINDOW_CORR}d — Bitcoin vs Activos', fontweight='bold')
axes[0].set_ylabel('Correlación')
axes[0].set_ylim(-1, 1)
axes[0].legend(fontsize=9)

# Heatmap correlación estática por año
retornos_btc_latam = retornos[['BTC-USD', 'ETH-USD', 'BNB-USD', 'ILF', 'GLD']]
corr_completa = retornos_btc_latam.corr()
sns.heatmap(corr_completa, annot=True, fmt='.2f', cmap='RdYlGn',
             vmin=-1, vmax=1, center=0, square=True, ax=axes[1])
axes[1].set_title('Matriz de Correlación Completa (2020-2024)', fontweight='bold')

plt.tight_layout()
plt.savefig('img/04_correlacion_dinamica.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 ¿BTC es un activo descorrelacionado de LATAM?')
corr_btc_ilf = retornos['BTC-USD'].corr(retornos['ILF'])
corr_btc_gld = retornos['BTC-USD'].corr(retornos['GLD'])
print(f'   Correlación BTC-ILF: {corr_btc_ilf:.3f}')
print(f'   Correlación BTC-GLD: {corr_btc_gld:.3f}')
if abs(corr_btc_ilf) < 0.3:
    print('   → BTC tiene baja correlación con LATAM → potencial diversificador')
else:
    print('   → BTC muestra correlación significativa con LATAM en este período')

## 6. BTC como Cobertura vs Inflación Bolivia

In [ ]:
# ─── BTC VS INFLACIÓN BOLIVIA ────────────────────────────────────────────────
# Comparación del poder adquisitivo: ¿qué le pasó a $1000 invertidos en BTC
# vs $1000 dejados en bolivianos desde enero 2020?

# Inflación acumulada Bolivia (aproximada por datos Banco Mundial)
# 2020: 0.67%, 2021: 0.74%, 2022: 1.74%, 2023: 2.28%, 2024: ~2.5% (estimado)
inflacion_anual_bo = {2020: 0.0067, 2021: 0.0074, 2022: 0.0174, 2023: 0.0228, 2024: 0.025}

# Poder adquisitivo perdido por inflación (base 1000 USD equivalente)
inversion_inicial = 1000
fechas_anuales = pd.date_range('2020-01-01', '2025-01-01', freq='AS')
poder_adq = [inversion_inicial]
for anio in range(2020, 2025):
    inf = inflacion_anual_bo.get(anio, 0.025)
    poder_adq.append(poder_adq[-1] / (1 + inf))

# BTC: valor de $1000 invertidos en enero 2020
btc_2020 = precios['BTC-USD']['2020-01-02']
btc_valor = (precios['BTC-USD'] / btc_2020 * inversion_inicial).resample('AS').first()

fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(btc_valor.index, btc_valor.values, color='#F7931A', lw=2.5,
         marker='o', ms=6, label='$1,000 invertidos en BTC')
ax.plot(fechas_anuales[:len(poder_adq)], poder_adq, color='#E74C3C', lw=2.5,
         marker='s', ms=6, linestyle='--', label='$1,000 en bolivianos (poder adq. real)')
ax.axhline(inversion_inicial, color='gray', linestyle=':', lw=1.5, label='Inversión inicial: $1,000')

ax.fill_between(btc_valor.index,
                  [inversion_inicial]*len(btc_valor),
                  btc_valor.values,
                  where=btc_valor.values >= inversion_inicial,
                  alpha=0.1, color='#27AE60')

ax.set_title('$1,000 en BTC vs Poder Adquisitivo en Bolivianos\n(Inflación acumulada Bolivia 2020-2024)',
              fontweight='bold')
ax.set_ylabel('Valor en USD')
ax.set_yscale('log')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('img/05_btc_vs_inflacion_bo.png', dpi=150, bbox_inches='tight')
plt.show()

ultimo_btc = btc_valor.dropna().iloc[-1]
ultimo_poder = poder_adq[-1]
print(f'\n📊 Comparación final (al cierre del período):')
print(f'   $1,000 en BTC → ${ultimo_btc:,.0f} ({(ultimo_btc/inversion_inicial-1)*100:+.1f}%)')
print(f'   $1,000 en BOB → ${ultimo_poder:,.2f} en poder adquisitivo ({(ultimo_poder/inversion_inicial-1)*100:+.1f}%)')
print(f'\n⚠️  Nota: BTC tuvo múltiples caídas del -70% en el mismo período.')
print(f'   Alta volatilidad = alto riesgo. No es una cobertura estable.')

In [ ]:
# ─── RESUMEN EJECUTIVO ────────────────────────────────────────────────────────
retorno_anual_btc = retornos['BTC-USD'].mean() * 252 * 100
vol_anual_btc     = retornos['BTC-USD'].std()  * np.sqrt(252) * 100
sharpe_btc        = (retorno_anual_btc/100 - 0.045) / (vol_anual_btc/100)

print('=' * 60)
print('RESUMEN — BITCOIN Y CRIPTO LATAM')
print('=' * 60)
print(f'\nPeríodo: {INICIO} → {FIN}')
print(f'\nBitcoin:')
print(f'  Retorno anual medio:  {retorno_anual_btc:+.1f}%')
print(f'  Volatilidad anual:     {vol_anual_btc:.1f}%')
print(f'  Sharpe Ratio:          {sharpe_btc:.3f}')
print(f'  Drawdown máximo:       {calcular_drawdown(precios["BTC-USD"]).min():.1f}%')
print(f'\nVs Oro (GLD):')
ret_gld = retornos['GLD'].mean() * 252 * 100
vol_gld = retornos['GLD'].std()  * np.sqrt(252) * 100
print(f'  Retorno anual medio:  {ret_gld:+.1f}%')
print(f'  Volatilidad anual:     {vol_gld:.1f}%')
print(f'  Sharpe Ratio:          {(ret_gld/100-0.045)/(vol_gld/100):.3f}')
print(f'\n💡 Conclusión para inversor boliviano:')
print(f'   BTC ofrece retornos muy superiores pero con volatilidad extrema.')
print(f'   Para cobertura de inflación boliviana, el oro es más estable.')
print(f'   BTC funciona mejor como posición especulativa de largo plazo (HODL).')